In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import glob
import os
from datetime import datetime, timedelta
from matplotlib.gridspec import GridSpec

Plot spatial maps of temperature with wind vectors on top for two snapshots on 13 and 17 January 2017 at 15:00 for Default(Bulk+NoUCM), LCZ, WSF-MB and Geoscape - Figure 19

In [ ]:
#2x2 Figure: Temperature with wind vectors on top for Jan 13 15:00 and Jan 17 15:00
# Directory paths for different scenarios
paths = {
    "Default(Bulk+NoUCM)": "/g/data/fy29/mf9078/run_WRF/etamodified/output/0out/selected/",
    "LCZ": "/g/data/fy29/mf9078/run_WRF/etamodified/output/1out/selected/",
    "WSF-MB": "/g/data/fy29/mf9078/run_WRF/etamodified/output/3out/selected/",
    "Geoscape": "/g/data/fy29/mf9078/run_WRF/etamodified/output/4out/selected/"
}

# Domain boundaries
lat_min, lat_max = -34.42, -33.3
lon_min, lon_max = 150.238, 151.55

lat_min_Syd, lat_max_Syd = -34.14, -33.55
lon_min_Syd, lon_max_Syd = 150.57, 151.37

# Output directory
output_dir = "/g/data/gb02/mf9078/plots/final/paperFigs"
os.makedirs(output_dir, exist_ok=True)

# Get list of files from first scenario (assuming all have same time steps)
first_scenario_path = list(paths.values())[0]
file_pattern = os.path.join(first_scenario_path, "wrfout_d02_2017-01-*")
reference_files = sorted(glob.glob(file_pattern))

print(f"Found {len(reference_files)} time steps to process")

# Process each time step
for file_idx, ref_file_path in enumerate(reference_files):
    try:
        # Extract filename for this time step
        filename = os.path.basename(ref_file_path)
        
        # Extract timestamp from filename and convert to AEST
        timestamp_utc = datetime.strptime(filename[11:], "%Y-%m-%d_%H:%M:%S")
        timestamp_aest = timestamp_utc + timedelta(hours=10)
        
        print(f"Processing time step {file_idx + 1}/{len(reference_files)}: {timestamp_aest.strftime('%Y-%m-%d %H:%M AEST')}")
        
        # Dictionary to store file paths for all scenarios
        scenario_files = {}
        data_dict = {}
        
        # Load data from all scenarios for this time step
        for scenario_name, scenario_path in paths.items():
            file_path = os.path.join(scenario_path, filename)
            
            if not os.path.exists(file_path):
                print(f"Warning: File not found: {file_path}")
                continue
                
            scenario_files[scenario_name] = file_path
            
        # Load datasets
        ds0 = xr.open_dataset(scenario_files["Default(Bulk+NoUCM)"])
        ds1 = xr.open_dataset(scenario_files["LCZ"])
        ds2 = xr.open_dataset(scenario_files["WSF-MB"])
        ds3 = xr.open_dataset(scenario_files["Geoscape"])
        
        # Get coordinates and select first time step (if there's a time dimension)
        XLAT = ds1["XLAT"].isel(Time=0) if "Time" in ds1["XLAT"].dims else ds1["XLAT"]
        XLONG = ds1["XLONG"].isel(Time=0) if "Time" in ds1["XLONG"].dims else ds1["XLONG"]
        LU_INDEX = ds1["LU_INDEX"].isel(Time=0) if "Time" in ds1["LU_INDEX"].dims else ds1["LU_INDEX"]
        
        # Create masks
        lat_mask = (XLAT >= lat_min) & (XLAT <= lat_max)
        lon_mask = (XLONG >= lon_min) & (XLONG <= lon_max)
        lat_mask_Syd = (XLAT >= lat_min_Syd) & (XLAT <= lat_max_Syd)
        lon_mask_Syd = (XLONG >= lon_min_Syd) & (XLONG <= lon_max_Syd)
        lu_mask = (LU_INDEX >= 51) & (LU_INDEX <= 60)
        Syd_mask = lat_mask_Syd & lon_mask_Syd & lu_mask
        data_mask = lat_mask & lon_mask

        # Create a binary mask for the boundary
        mask_binary = Syd_mask.astype(int)

        # Get data and select first time step if needed for Bulk
        t2_data0 = ds0["T2"].isel(Time=0) if "Time" in ds0["T2"].dims else ds0["T2"]
        u10_data0 = ds0["U10"].isel(Time=0) if "Time" in ds0["U10"].dims else ds0["U10"]
        v10_data0 = ds0["V10"].isel(Time=0) if "Time" in ds0["V10"].dims else ds0["V10"]
        
        # Get data and select first time step if needed for LCZ
        t2_data1 = ds1["T2"].isel(Time=0) if "Time" in ds1["T2"].dims else ds1["T2"]
        u10_data1 = ds1["U10"].isel(Time=0) if "Time" in ds1["U10"].dims else ds1["U10"]
        v10_data1 = ds1["V10"].isel(Time=0) if "Time" in ds1["V10"].dims else ds1["V10"]
       
        # Get data and select first time step if needed for WSF3DMSB
        t2_data2 = ds2["T2"].isel(Time=0) if "Time" in ds2["T2"].dims else ds2["T2"]
        u10_data2 = ds2["U10"].isel(Time=0) if "Time" in ds2["U10"].dims else ds2["U10"]
        v10_data2 = ds2["V10"].isel(Time=0) if "Time" in ds2["V10"].dims else ds2["V10"]
        
        # Get data and select first time step if needed for Geoscape
        t2_data3 = ds3["T2"].isel(Time=0) if "Time" in ds3["T2"].dims else ds3["T2"]
        u10_data3 = ds3["U10"].isel(Time=0) if "Time" in ds3["U10"].dims else ds3["U10"]
        v10_data3 = ds3["V10"].isel(Time=0) if "Time" in ds3["V10"].dims else ds3["V10"]
        
        scenario_data = {
            "Default(Bulk+NoUCM)": {"t2": t2_data0-273.15, "u10": u10_data0, "v10": v10_data0},
            "LCZ": {"t2": t2_data1-273.15, "u10": u10_data1, "v10": v10_data1},
            "WSF-MB": {"t2": t2_data2-273.15, "u10": u10_data2, "v10": v10_data2},
            "Geoscape": {"t2": t2_data3-273.15, "u10": u10_data3, "v10": v10_data3}
        }
        
        # Define layout order: Top row: Bulk, Geoscape; Bottom row: LCZ, WSF-MB
        layout_order = [["Default(Bulk+NoUCM)", "Geoscape"], ["LCZ", "WSF-MB"]]
        
        # Solution: Use gridspec for 2x2 layout with colorbar
                
        fig = plt.figure(figsize=(20, 16))  # Square-ish figure for 2x2 layout
        
        # Create gridspec with 2 rows, 3 columns (2 for plots + 1 for colorbar)
        gs = GridSpec(2, 3, width_ratios=[1, 1, 0.05], 
                     wspace=0.1, hspace=0.15, figure=fig)
        
        # Create subplots
        axs = []
        for row in range(2):
            axs_row = []
            for col in range(2):
                axs_row.append(fig.add_subplot(gs[row, col], projection=ccrs.PlateCarree()))
            axs.append(axs_row)
        
        # Create colorbar axis spanning both rows
        cbar_ax = fig.add_subplot(gs[:, 2])

        # Store mesh for colorbar
        mesh_for_colorbar = None
        
        # Plot each scenario
        for row in range(2):
            for col in range(2):
                scenario = layout_order[row][col]
                ax = axs[row][col]
                
                # Set map extent and features
                ax.set_extent([lon_min, lon_max, lat_min, lat_max])
                ax.add_feature(cfeature.GSHHSFeature(scale="high"))
                ax.add_feature(cfeature.BORDERS, linestyle="--", edgecolor="gray")
                ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.3)
                
                # Use contour to draw the boundary
                boundary = ax.contour(XLONG, XLAT, mask_binary, levels=[0.5], 
                                      colors='black', linewidths=2, linestyles='-',
                                      transform=ccrs.PlateCarree())

                # Plot temperature data
                mesh = ax.pcolormesh(XLONG, XLAT, scenario_data[scenario]["t2"], 
                                   transform=ccrs.PlateCarree(), 
                                   cmap="jet", alpha=0.8, shading='nearest', 
                                   vmin=20, vmax=40)
                
                # Store mesh for colorbar (use the last one)
                mesh_for_colorbar = mesh

                # Add wind vectors
                skip = 8
                Q = ax.quiver(
                    XLONG[::skip, ::skip], XLAT[::skip, ::skip], 
                    scenario_data[scenario]["u10"][::skip, ::skip], 
                    scenario_data[scenario]["v10"][::skip, ::skip], 
                    scale=15, scale_units='inches', color="black", width=0.0025
                )

                # Add quiver key only to the top-left subplot
                if row == 0 and col == 0:
                    qk = ax.quiverkey(Q, -0.07, 1.04, 2, r'$2 \frac{m}{s}$', labelpos='E',
                                     coordinates='axes', color='black', fontproperties={'size': 32})
                
                # Add scenario title above each subplot
                ax.set_title(scenario, fontsize=36, pad=20)
                
                # Add y-label only to the left column
                if col == 0:
                    ax.set_ylabel("Latitude", fontsize=28)
                
                # Add x-label only to the bottom row
                if row == 1:
                    ax.set_xlabel("Longitude", fontsize=28)
                
                # Add gridlines
                gl = ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False, alpha=0.2)
                gl.xlabel_style = {'size': 16}
                #gl.ylabel_style = {'size': 16}
                gl.top_labels = False
                gl.right_labels = False
                #gl.left_labels = False

                # Only show y-axis labels (left labels) for the left column
                if col != 0:
                    gl.left_labels = False
                
                # Only show x-axis labels (bottom labels) for the bottom row
                if row != 1:
                    gl.bottom_labels = False
        
        # Add colorbar to the separate axis
        cbar = plt.colorbar(mesh_for_colorbar, cax=cbar_ax, extend='both')
        cbar.set_label("2-m Air Temperature (°C)", rotation=90, fontsize=32)
        cbar.ax.tick_params(labelsize=28)
        
        # Set main title with AEST time
        plt.suptitle(f"{timestamp_aest.strftime('%Y-%m-%d %H:%M')}", 
                     fontsize=36, y=0.96)
        
        # Save figure
        filename_out = f"hourly_heat_{timestamp_aest.strftime('%Y%m%d_%H%M')}_AEST_withdefault_.png"
        output_path = os.path.join(output_dir, filename_out)
        
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()  # Close to free memory
        
        # Close datasets
        ds0.close()
        ds1.close()
        ds2.close()
        ds3.close()
        
        print(f"Saved: {filename_out}")
        
    except Exception as e:
        print(f"Error processing file {ref_file_path}: {e}")
        continue

print("Processing complete!")

Found 2 time steps to process
Processing time step 1/2: 2017-01-13 15:00 AEST
Saved: hourly_heat_20170113_1500_AEST_withdefault_.png
Processing time step 2/2: 2017-01-17 15:00 AEST
Saved: hourly_heat_20170117_1500_AEST_withdefault_.png
Processing complete!
